# 01 — Data Preparation and Volatility Construction

This notebook builds the complete analysis-ready panel for the dissertation from the raw Bloomberg exports.

**Scope, following the finalised research design:**

- **FX targets (four core pairs only):** EURUSD, USDJPY, AUDUSD, USDCAD. 
- **Rates channel — primary:** 2-year yield differentials (US–DE for EURUSD, US–JP for USDJPY).
- **Rates channel — robustness:** 10-year yield differentials (same two pairs), and the individual 2-year yields (US2Y, DE2Y, JP2Y).
- **Commodity channel:** Brent (primary oil benchmark), Gold, Silver. 
- **Volatility indices:** JPM_FXVOL (primary FX implied vol) and VIX (broad risk-sentiment regime classifier). CVIX is dropped as redundant with JPM_FXVOL — both are FX-implied-vol measures and only one is needed.
- **No yield-curve slopes.** Slope and slope-differential variables answered a different question (term-structure shape) that is not part of this design and are not constructed.

**Volatility proxies constructed for every rates/commodity/FX innovation series:**
1. Absolute innovation (model-free, descriptive workhorse)
2. Squared innovation (model-free, robustness)
3. GARCH(1,1) conditional volatility (primary analysis proxy)
4. EGARCH(1,1) conditional volatility with a leverage term (used where asymmetry is present)

**Missing-data policy:** holidays and non-trading days are set to `NaN`, never forward-filled — a forward-filled value would create an artificial zero-change day and contaminate the volatility proxies. Two panels are exported: a **wide panel** that keeps each series' own missing pattern (for pairwise analysis, so one series' gaps never shrink another pair's usable sample), and a **strict common-date panel** restricted to dates where every retained variable is observed (for analyses that require one shared sample, e.g. a joint VAR).

In [69]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
from arch import arch_model

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

PROJECT_NAME = "Volatility_Project"

def find_project_root(name, start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start] + list(start.parents):
        if candidate.name == name:
            return candidate
    raise FileNotFoundError(
        f"Could not find a folder named '{name}' above {start}. "
        f"Hard-code BASE below instead."
    )

try:
    BASE = find_project_root(PROJECT_NAME)
except FileNotFoundError as e:
    print(e)
    # fallback: hard-code your absolute path here if the search above fails, e.g.:
    # BASE = Path("/Users/nancyshi/Downloads/2026 summer project/Volatility_Project")
    raise

RAW_DIR       = BASE / "data" / "raw"
PROCESSED_DIR = BASE / "data" / "processed"
FIGURES_DIR   = BASE / "figures"
TABLES_DIR    = BASE / "tables"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

FX_FILE    = RAW_DIR / "fx_daily_2015_2024.xlsx"
BOND_FILE  = RAW_DIR / "bond_2015_2024.xlsx"
COMM_FILE  = RAW_DIR / "commodities_2015_2024.xlsx"
VOL_FILE   = RAW_DIR / "Volatility_2015_2024.xlsx"
# -----------------------------------------------------------------------------

print("Project root:", BASE)
print("Raw directory:", RAW_DIR)
print("Processed output directory:", PROCESSED_DIR)

Project root: /Users/nancyshi/Desktop/2026 summer project/Volatility_Project
Raw directory: /Users/nancyshi/Desktop/2026 summer project/Volatility_Project/data/raw
Processed output directory: /Users/nancyshi/Desktop/2026 summer project/Volatility_Project/data/processed


## 1. Data dictionary

Only the series retained in the final design are listed. Each Bloomberg export stores every ticker as an independent (Date, Value) block, read from the `fixed_values` sheet by column position.

In [70]:
data_dictionary = pd.DataFrame(
    [
        ["EURUSD", "EURUSD Curncy", "FX",         "Core FX pair — rates channel (US-DE)"],
        ["USDJPY", "USDJPY Curncy", "FX",         "Core FX pair — rates channel (US-JP)"],
        ["AUDUSD", "AUDUSD Curncy", "FX",         "Core FX pair — commodity channel (metals)"],
        ["USDCAD", "USDCAD Curncy", "FX",         "Core FX pair — commodity channel (oil)"],
        ["DXY", "DXY Curncy", "FX", "Common-factor control for conditional GC and TE"],

        ["US2Y",   "USGG2YR Index", "Bond",       "2Y Treasury yield — primary rates source"],
        ["US10Y",  "USGG10YR Index","Bond",       "10Y Treasury yield — robustness"],
        ["DE2Y",   "GDBR2 Index",   "Bond",       "2Y German yield — primary rates source"],
        ["DE10Y",  "GDBR10 Index",  "Bond",       "10Y German yield — robustness"],
        ["JP2Y",   "GJGB2 Index",   "Bond",       "2Y Japanese yield — primary rates source"],
        ["JP10Y",  "GJGB10 Index",  "Bond",       "10Y Japanese yield — robustness"],

        ["Brent",  "CO1 Comdty",    "Commodity",  "Primary oil benchmark (USDCAD channel)"],
        ["Gold",   "XAU Curncy",    "Commodity",  "Primary metals benchmark (AUDUSD channel)"],
        ["Silver", "XAG Curncy",    "Commodity",  "Secondary metals benchmark (AUDUSD channel)"],

        ["JPM_FXVOL", "JPM G7 FX Volatility", "Volatility", "Primary FX-implied volatility"],
        ["VIX",       "VIX Index",            "Volatility", "Broad risk sentiment — regime classifier"],
    ],
    columns=["Variable", "Bloomberg Ticker", "Asset Class", "Role in design"],
)

data_dictionary


,Variable,Bloomberg Ticker,Asset Class,Role in design
0,EURUSD,EURUSD Curncy,FX,Core FX pair — rates channel (US-DE)
1,USDJPY,USDJPY Curncy,FX,Core FX pair — rates channel (US-JP)
2,AUDUSD,AUDUSD Curncy,FX,Core FX pair — commodity channel (metals)
3,USDCAD,USDCAD Curncy,FX,Core FX pair — commodity channel (oil)
4,DXY,DXY Curncy,FX,Common-factor control for conditional GC and TE
5,US2Y,USGG2YR Index,Bond,2Y Treasury yield — primary rates source
6,US10Y,USGG10YR Index,Bond,10Y Treasury yield — robustness
7,DE2Y,GDBR2 Index,Bond,2Y German yield — primary rates source
8,DE10Y,GDBR10 Index,Bond,10Y German yield — robustness
9,JP2Y,GJGB2 Index,Bond,2Y Japanese yield — primary rates source


## 2. Read raw Bloomberg blocks

Each ticker in a Bloomberg export occupies a (date, value) column pair. The function below reads one such pair by position and returns a clean two-column frame. Only the pairs needed for the retained series are read — WTI, CVIX, GBPUSD, USDCHF and DXY are skipped here rather than read and discarded later, so nothing unused ever enters the pipeline.

In [71]:
def read_bdh_block(raw, date_col, value_col, name):
    """Read one (date, value) Bloomberg column pair by position and return
    a tidy two-column frame: Date, <name>."""
    out = raw.iloc[1:, [date_col, value_col]].copy()
    out.columns = ["Date", name]
    out["Date"] = pd.to_datetime(out["Date"])
    out[name] = pd.to_numeric(out[name], errors="coerce")
    return out.dropna(subset=["Date"]).reset_index(drop=True)


# --- FX: columns are (date, value) pairs at (0,1), (2,3), (4,5), (6,7); ---
# --- GBPUSD/USDCHF/DXY pairs at (8,9)/(10,11)/(12,13) exist in the file but are skipped. ---
fx_raw = pd.read_excel(FX_FILE, sheet_name="fixed_values", header=None)
eurusd = read_bdh_block(fx_raw, 0, 1, "EURUSD")
usdjpy = read_bdh_block(fx_raw, 2, 3, "USDJPY")
audusd = read_bdh_block(fx_raw, 4, 5, "AUDUSD")
usdcad = read_bdh_block(fx_raw, 6, 7, "USDCAD")
dxy    = read_bdh_block(fx_raw, 12, 13, "DXY")

# --- Bonds: all six yields are read (2Y primary, 10Y robustness) ---
bond_raw = pd.read_excel(BOND_FILE, sheet_name="fixed_values", header=None)
us2y  = read_bdh_block(bond_raw, 0,  1,  "US2Y")
us10y = read_bdh_block(bond_raw, 2,  3,  "US10Y")
de2y  = read_bdh_block(bond_raw, 4,  5,  "DE2Y")
de10y = read_bdh_block(bond_raw, 6,  7,  "DE10Y")
jp2y  = read_bdh_block(bond_raw, 8,  9,  "JP2Y")
jp10y = read_bdh_block(bond_raw, 10, 11, "JP10Y")

# --- Commodities: WTI pair at (0,1) is skipped; only Brent, Gold, Silver read ---
commodity_raw = pd.read_excel(COMM_FILE, sheet_name="fixed_values", header=None)
brent  = read_bdh_block(commodity_raw, 2, 3, "Brent")
gold   = read_bdh_block(commodity_raw, 4, 5, "Gold")
silver = read_bdh_block(commodity_raw, 6, 7, "Silver")

# --- Volatility: CVIX pair at (2,3) is skipped; only JPM_FXVOL and VIX read ---
volatility_raw = pd.read_excel(VOL_FILE, sheet_name="fixed_values", header=None)
jpm_vol = read_bdh_block(volatility_raw, 0, 1, "JPM_FXVOL")
vix     = read_bdh_block(volatility_raw, 4, 5, "VIX")

all_series = [eurusd, usdjpy, audusd, usdcad, dxy,
              us2y, us10y, de2y, de10y, jp2y, jp10y,
              brent, gold, silver,
              jpm_vol, vix]

for s in all_series:
    print(f"{s.columns[1]:10s}  n={len(s):5d}  {s['Date'].min().date()} -> {s['Date'].max().date()}")

EURUSD      n= 2609  2015-01-01 -> 2024-12-31
USDJPY      n= 2609  2015-01-01 -> 2024-12-31
AUDUSD      n= 2609  2015-01-01 -> 2024-12-31
USDCAD      n= 2609  2015-01-01 -> 2024-12-31
DXY         n= 2602  2015-01-01 -> 2024-12-31
US2Y        n= 2608  2015-01-01 -> 2024-12-31
US10Y       n= 2608  2015-01-01 -> 2024-12-31
DE2Y        n= 2603  2015-01-01 -> 2024-12-31
DE10Y       n= 2604  2015-01-01 -> 2024-12-31
JP2Y        n= 2608  2015-01-01 -> 2024-12-31
JP10Y       n= 2607  2015-01-01 -> 2024-12-31
Brent       n= 2582  2015-01-02 -> 2024-12-31
Gold        n= 2595  2015-01-01 -> 2024-12-31
Silver      n= 2599  2015-01-01 -> 2024-12-31
JPM_FXVOL   n= 2609  2015-01-01 -> 2024-12-31
VIX         n= 2535  2015-01-02 -> 2024-12-31


## 3. Clean non-trading days and stale holiday observations

Two distinct problems need separate treatment.

First, 1 January and 25 December appear in every Bloomberg export as non-trading placeholder rows and are dropped from all series.

Second, sovereign bond yields carry **stale carry-forward values** on each country's own national holidays (days the local market is closed but Bloomberg's generic index repeats the last quote). These are set to `NaN` — not dropped from the row, since other series may still have genuine data that day — using the `holidays` package matched to each country's calendar. Value-repetition detection is deliberately not used, because it would incorrectly flag genuine flat-yield days during BoJ yield-curve-control, which are real data rather than staleness.

In [72]:
def remove_shared_non_trading_days(df):
    """Drop 1 Jan and 25 Dec placeholder rows from a single-series frame."""
    return df[
        ~(((df["Date"].dt.month == 1) & (df["Date"].dt.day == 1)) |
          ((df["Date"].dt.month == 12) & (df["Date"].dt.day == 25)))
    ].reset_index(drop=True)


all_series = [remove_shared_non_trading_days(s) for s in all_series]
(eurusd, usdjpy, audusd, usdcad, dxy,
 us2y, us10y, de2y, de10y, jp2y, jp10y,
 brent, gold, silver,
 jpm_vol, vix) = all_series



In [73]:
import holidays

def null_national_holidays(df, value_col, country_calendar, years=range(2015, 2025)):
    """Set the yield to NaN on that country's own national holidays,
    rather than dropping the row (other series may still be valid that day)."""
    cal_dates = set(country_calendar(years=years).keys())
    mask = df["Date"].dt.date.isin(cal_dates)
    df = df.copy()
    df.loc[mask, value_col] = np.nan
    n_nulled = mask.sum()
    print(f"{value_col:8s}: {n_nulled} national-holiday observations set to NaN")
    return df

us2y  = null_national_holidays(us2y,  "US2Y",  holidays.US)
us10y = null_national_holidays(us10y, "US10Y", holidays.US)
de2y  = null_national_holidays(de2y,  "DE2Y",  holidays.DE)
de10y = null_national_holidays(de10y, "DE10Y", holidays.DE)
jp2y  = null_national_holidays(jp2y,  "JP2Y",  holidays.JP)
jp10y = null_national_holidays(jp10y, "JP10Y", holidays.JP)

US2Y    : 89 national-holiday observations set to NaN
US10Y   : 89 national-holiday observations set to NaN
DE2Y    : 58 national-holiday observations set to NaN
DE10Y   : 58 national-holiday observations set to NaN
JP2Y    : 138 national-holiday observations set to NaN
JP10Y   : 137 national-holiday observations set to NaN


The holiday counts scale with how many public holidays each country observes on top of weekends — Japan (138–139 days) well above Germany (58 days) and the US (89 days). This is expected: it reflects each country's own calendar, not a data quality problem. These are set to `NaN` in place rather than dropped as rows, so a US or German holiday does not remove an otherwise-valid Japanese observation on the same date.

## 4. Construct returns, yield changes, and differentials

FX and commodities are converted to daily log returns (%). Bond yields are converted to daily changes in basis points. Only the 2Y and 10Y differentials actually used in the design are built — US-DE and US-JP, at both tenors — since only EURUSD and USDJPY have a theoretically grounded rates channel. No DE-JP differential is constructed, as no FX pair in this study maps onto it.

In [74]:
def add_log_return(df, price_col, return_col):
    df = df.copy()
    df[return_col] = 100 * np.log(df[price_col] / df[price_col].shift(1))
    return df

def add_yield_change(df, yield_col, change_col):
    df = df.copy()
    df[change_col] = 100 * df[yield_col].diff()   # yield in %, change in bp
    return df

eurusd = add_log_return(eurusd, "EURUSD", "EURUSD_Return")
usdjpy = add_log_return(usdjpy, "USDJPY", "USDJPY_Return")
audusd = add_log_return(audusd, "AUDUSD", "AUDUSD_Return")
usdcad = add_log_return(usdcad, "USDCAD", "USDCAD_Return")
dxy = add_log_return(dxy, "DXY", "DXY_Return")

brent  = add_log_return(brent,  "Brent",  "Brent_Return")
gold   = add_log_return(gold,   "Gold",   "Gold_Return")
silver = add_log_return(silver, "Silver", "Silver_Return")

us2y  = add_yield_change(us2y,  "US2Y",  "US2Y_Change_bp")
us10y = add_yield_change(us10y, "US10Y", "US10Y_Change_bp")
de2y  = add_yield_change(de2y,  "DE2Y",  "DE2Y_Change_bp")
de10y = add_yield_change(de10y, "DE10Y", "DE10Y_Change_bp")
jp2y  = add_yield_change(jp2y,  "JP2Y",  "JP2Y_Change_bp")
jp10y = add_yield_change(jp10y, "JP10Y", "JP10Y_Change_bp")

jpm_vol = add_log_return(jpm_vol, "JPM_FXVOL", "JPM_FXVOL_Change")  # level change, kept simple
vix     = add_log_return(vix,     "VIX",       "VIX_Change")

## 5. Align series and build the wide panel

All series are merged on their actual observation dates using an outer join, so each series keeps its own missing-data pattern rather than being forced onto a common index at this stage. Yield differentials — the primary rates-channel source — are computed after alignment, since a differential is only defined where both legs are observed.

In [75]:
series_list = [eurusd, usdjpy, audusd, usdcad, dxy,
               us2y, us10y, de2y, de10y, jp2y, jp10y,
               brent, gold, silver,
               jpm_vol, vix]

wide = pd.concat([s.set_index("Date") for s in series_list], axis=1)
wide = wide.sort_index().reset_index()

print("Wide panel before differentials:", wide.shape)
wide.head()

Wide panel before differentials: (2595, 33)


,Date,EURUSD,EURUSD_Return,USDJPY,USDJPY_Return,AUDUSD,AUDUSD_Return,USDCAD,USDCAD_Return,DXY,...,Brent,Brent_Return,Gold,Gold_Return,Silver,Silver_Return,JPM_FXVOL,JPM_FXVOL_Change,VIX,VIX_Change
0,2015-01-02,1.2002,NaN,120.5000,NaN,0.8089,NaN,1.1785,NaN,91.0800,...,56.4200,NaN,"1,188.3900",NaN,15.7465,NaN,9.8200,NaN,17.7900,NaN
1,2015-01-05,1.1933,-0.5766,119.6400,-0.7163,0.8083,-0.0742,1.1763,-0.1869,91.3780,...,53.1100,-6.0458,"1,204.8600",1.3764,16.1885,2.7683,9.7100,-1.1265,19.9200,11.3088
2,2015-01-06,1.1890,-0.3610,118.3900,-1.0503,0.8084,0.0124,1.1836,0.6187,91.4990,...,51.1000,-3.8581,"1,218.5800",1.1323,16.5365,2.1269,9.7900,0.8205,21.1200,5.8496
3,2015-01-07,1.1839,-0.4299,119.2600,0.7322,0.8079,-0.0619,1.1815,-0.1776,91.8900,...,51.1500,0.0978,"1,211.4100",-0.5901,16.5354,-0.0067,9.9200,1.3191,19.3100,-8.9597
4,2015-01-08,1.1793,-0.3893,119.6600,0.3348,0.8123,0.5431,1.1831,0.1353,92.3680,...,50.9600,-0.3721,"1,208.7900",-0.2165,16.3651,-1.0353,9.6800,-2.4491,17.0100,-12.6822


In [76]:
# 2Y differentials (primary rates source)
wide["US_DE_2Y_Diff"] = wide["US2Y"] - wide["DE2Y"]
wide["US_JP_2Y_Diff"] = wide["US2Y"] - wide["JP2Y"]

# 10Y differentials (robustness: tenor sensitivity)
wide["US_DE_10Y_Diff"] = wide["US10Y"] - wide["DE10Y"]
wide["US_JP_10Y_Diff"] = wide["US10Y"] - wide["JP10Y"]

# Daily changes in each differential, in basis points
for diff_col, change_col in [
    ("US_DE_2Y_Diff",  "US_DE_2Y_Diff_Change_bp"),
    ("US_JP_2Y_Diff",  "US_JP_2Y_Diff_Change_bp"),
    ("US_DE_10Y_Diff", "US_DE_10Y_Diff_Change_bp"),
    ("US_JP_10Y_Diff", "US_JP_10Y_Diff_Change_bp"),
]:
    wide[change_col] = 100 * wide[diff_col].diff()

print("Differentials constructed:",
      ["US_DE_2Y_Diff", "US_JP_2Y_Diff", "US_DE_10Y_Diff", "US_JP_10Y_Diff"])

Differentials constructed: ['US_DE_2Y_Diff', 'US_JP_2Y_Diff', 'US_DE_10Y_Diff', 'US_JP_10Y_Diff']


## 6. Volatility proxy construction

Every innovation series relevant to the design gets three representations: model-free absolute and squared innovations, GARCH(1,1) conditional volatility, and EGARCH(1,1) conditional volatility with a leverage term. `JPM_FXVOL` and `VIX` are already implied volatilities and are kept as levels — they are not passed through GARCH.

The rates block covers both the primary differential source and the robustness sources (10Y differentials, individual 2Y yields) so the transmission analysis can later show it does not hinge on any one specification.

In [77]:
innovation = {
    # FX targets
    "EURUSD": "EURUSD_Return", "USDJPY": "USDJPY_Return",
    "AUDUSD": "AUDUSD_Return", "USDCAD": "USDCAD_Return",
    # Broad-dollar common-factor control
    "DXY": "DXY_Return",
    # Rates - primary (2Y differentials)
    "US_DE_2Y_Diff": "US_DE_2Y_Diff_Change_bp",
    "US_JP_2Y_Diff": "US_JP_2Y_Diff_Change_bp",
    # Rates - robustness (10Y differentials)
    "US_DE_10Y_Diff": "US_DE_10Y_Diff_Change_bp",
    "US_JP_10Y_Diff": "US_JP_10Y_Diff_Change_bp",
    # Rates - robustness (individual 2Y yields)
    "US2Y": "US2Y_Change_bp", "DE2Y": "DE2Y_Change_bp", "JP2Y": "JP2Y_Change_bp",
    # Commodities
    "Brent": "Brent_Return", "Gold": "Gold_Return", "Silver": "Silver_Return",
}
implied_vol = ["JPM_FXVOL", "VIX"]

missing_cols = [v for v in innovation.values() if v not in wide.columns]
assert not missing_cols, f"Missing expected columns: {missing_cols}"
print(f"{len(innovation)} innovation series mapped for volatility construction.")

15 innovation series mapped for volatility construction.


In [78]:
# Model-free proxies
for name, col in innovation.items():
    wide[f"{name}_abs"] = wide[col].abs()
    wide[f"{name}_sq"]  = wide[col] ** 2

print("Absolute and squared innovation proxies constructed.")

Absolute and squared innovation proxies constructed.


In [79]:
def fit_conditional_vol(series, vol="GARCH", asym=False):
    """Fit a (E)GARCH(1,1)-t on the innovation series and return conditional
    volatility on the original scale, plus the fitted result object."""
    s = series.dropna()
    o = 1 if asym else 0
    res = arch_model(s, mean="Constant", vol=vol, p=1, o=o, q=1,
                     dist="t", rescale=True).fit(disp="off")
    cond_vol = pd.Series(res.conditional_volatility / res.scale, index=s.index)
    return res, cond_vol


garch_persist = {}
for name, col in innovation.items():
    res, cv = fit_conditional_vol(wide[col], vol="GARCH", asym=False)
    wide.loc[cv.index, f"{name}_garchvol"] = cv
    garch_persist[name] = res.params["alpha[1]"] + res.params["beta[1]"]

pd.Series(garch_persist, name="alpha+beta (persistence)").round(4).to_frame()

,alpha+beta (persistence)
EURUSD,0.9939
USDJPY,0.9838
AUDUSD,0.9933
USDCAD,0.9955
DXY,0.9912
US_DE_2Y_Diff,1.0000
US_JP_2Y_Diff,1.0000
US_DE_10Y_Diff,0.9650
US_JP_10Y_Diff,0.9918
US2Y,1.0000


Persistence (α+β) sits close to one across nearly every series, and is effectively 1.0000 for every rates series — both the 2Y differentials and the individual yields. This confirms the near-integrated, slow-decaying volatility documented in the EDA notebook, and is the empirical basis for treating daily-frequency volatility transmission as a multi-day phenomenon rather than one that daily sampling would obscure.

In [80]:
rows = []
for name, col in innovation.items():
    g_res, _  = fit_conditional_vol(wide[col], vol="GARCH",  asym=False)
    e_res, ev = fit_conditional_vol(wide[col], vol="EGARCH", asym=True)
    wide.loc[ev.index, f"{name}_egarchvol"] = ev
    rows.append({
        "series": name,
        "persistence": garch_persist[name],
        "leverage_gamma": e_res.params["gamma[1]"],
        "gamma_pvalue": e_res.pvalues["gamma[1]"],
        "GARCH_BIC": g_res.bic,
        "EGARCH_BIC": e_res.bic,
        "preferred": "EGARCH" if e_res.bic < g_res.bic else "GARCH",
    })

model_summary = pd.DataFrame(rows).set_index("series")
model_summary.round(4)

,persistence,leverage_gamma,gamma_pvalue,GARCH_BIC,EGARCH_BIC,preferred
series,,,,,,
EURUSD,0.9939,-0.0085,0.4185,"3,353.3178","3,368.0704",GARCH
USDJPY,0.9838,-0.0246,0.0820,"3,752.0850","3,759.3500",GARCH
AUDUSD,0.9933,-0.0190,0.0446,"4,794.5550","4,804.3816",GARCH
USDCAD,0.9955,0.0307,0.0005,"3,067.0057","3,070.0866",GARCH
DXY,0.9912,0.0037,0.7482,"2,707.8570","2,724.6381",GARCH
US_DE_2Y_Diff,1.0000,0.0087,0.5173,"12,194.3015","12,198.7140",GARCH
US_JP_2Y_Diff,1.0000,-0.0143,0.3712,"12,036.8821","12,039.0304",GARCH
US_DE_10Y_Diff,0.9650,-0.0406,0.0597,"12,787.9505","12,797.6927",GARCH
US_JP_10Y_Diff,0.9918,-0.0049,0.6787,"13,300.1765","13,307.2383",GARCH


In [81]:
# ---------------------------------------------------------------------
# Primary volatility-shock variables used in GC and TE
# s_t = Δ log sigma_hat_t
# ---------------------------------------------------------------------

for name in innovation:
    vol = pd.to_numeric(wide[f"{name}_garchvol"], errors="coerce")
    vol = vol.where(vol > 0)
    wide[f"{name}_dlogvol"] = np.log(vol).diff()

print("Δ log GARCH volatility shocks constructed for all innovation series.")

Δ log GARCH volatility shocks constructed for all innovation series.


In [82]:
cols = [
    "EURUSD_dlogvol",
    "USDJPY_dlogvol",
    "AUDUSD_dlogvol",
    "USDCAD_dlogvol",
    "US_DE_2Y_Diff_dlogvol",
    "US_JP_2Y_Diff_dlogvol",
    "Brent_dlogvol",
    "Gold_dlogvol",
    "DXY_dlogvol",
]

wide[cols].head()

,EURUSD_dlogvol,USDJPY_dlogvol,AUDUSD_dlogvol,USDCAD_dlogvol,US_DE_2Y_Diff_dlogvol,US_JP_2Y_Diff_dlogvol,Brent_dlogvol,Gold_dlogvol,DXY_dlogvol
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,-0.0095,0.0078,-0.0141,-0.0161,-0.0239,-0.0337,0.0843,0.0069,-0.0115
3,-0.0154,0.0567,-0.0142,-0.0044,-0.0153,-0.0222,-0.0013,-0.0023,-0.0181
4,-0.0135,-0.0022,-0.0140,-0.0162,-0.0275,-0.0321,-0.0542,-0.0146,-0.0049


Reading the leverage column by asset class: FX and the rate differentials are uniformly preferred as plain GARCH, and where the leverage term is nominally significant (AUDUSD, USDCAD) the BIC comparison still favours the simpler symmetric model — asymmetry is not a robust FX feature at daily frequency. Commodities are the opposite case: all three are preferred as EGARCH, all three have a significant leverage term, and the sign is economically sensible — Brent negative (downside shocks raise volatility more, as in equities), Gold and Silver positive (upside shocks raise volatility more, consistent with safe-haven demand). This is the empirical basis for using the GARCH proxy as the primary volatility measure for FX and rates, and the EGARCH proxy for the commodity channel.

## 7. Quality checks

Basic checks on the wide panel before export: shape, sample period, duplicate dates, and how much each series is missing (a useful record of exactly what the holiday-nulling step removed).

In [83]:
print("Wide panel shape:", wide.shape)
print("Sample period:", wide["Date"].min().date(), "->", wide["Date"].max().date())
print("Duplicate dates:", wide["Date"].duplicated().sum())
print()
missing = wide.drop(columns="Date").isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print("Missing observations by column (non-zero only):")
missing.to_frame("n_missing")

Wide panel shape: (2595, 116)
Sample period: 2015-01-02 -> 2024-12-31
Duplicate dates: 0

Missing observations by column (non-zero only):


,n_missing
US_JP_10Y_Diff_dlogvol,586
US_JP_2Y_Diff_dlogvol,586
US_DE_2Y_Diff_dlogvol,427
US_DE_10Y_Diff_dlogvol,426
US_JP_2Y_Diff_Change_bp,405
...,...
EURUSD_egarchvol,1
USDJPY_Return,1
EURUSD_garchvol,1
USDCAD_garchvol,1


The largest missing counts belong to the differential and change-in-differential columns involving Japan, and they are considerably larger than the raw holiday count alone (e.g. `US_JP_2Y_Diff_Change_bp` at 405, versus 138 raw JP2Y holidays). This is the expected consequence of computing `.diff()` over a series with `NaN`-in-place holiday values: the day immediately following each holiday also becomes `NaN`, because its change requires the previous day's value. This is mathematically correct behaviour, not a defect, but it means any analysis using a Japan-linked differential should expect a meaningfully smaller usable sample than the raw yield series alone would suggest.

In [84]:
outlier_threshold = 8

flagged_rows = []
for name, col in innovation.items():
    s = wide[col].dropna()
    z = (s - s.mean()) / s.std()
    hits = z[z.abs() > outlier_threshold]
    for idx, zval in hits.items():
        flagged_rows.append({
            "series": name,
            "Date": wide.loc[idx, "Date"],
            "value": wide.loc[idx, col],
            "z_score": zval,
        })

outlier_check = pd.DataFrame(flagged_rows)

if outlier_check.empty:
    print(f"No observations exceed |z| > {outlier_threshold} in any series.")
else:
    print(f"{len(outlier_check)} observation(s) exceed |z| > {outlier_threshold}:")
    display(outlier_check.sort_values(["series", "Date"]).reset_index(drop=True))

10 observation(s) exceed |z| > 8:


,series,Date,value,z_score
0,Brent,2020-03-09,-27.5751,-10.9009
1,Brent,2020-04-21,-27.9761,-11.0594
2,DE2Y,2023-03-13,-40.7000,-10.1074
3,DE2Y,2023-03-15,-48.3000,-11.9903
4,JP2Y,2016-07-29,9.7000,8.2801
5,JP2Y,2020-03-27,10.1000,8.6223
6,JP2Y,2024-08-05,-11.0000,-9.4283
7,Silver,2020-08-11,-16.1186,-9.6252
8,US2Y,2023-03-13,-60.9800,-11.8945
9,US_JP_2Y_Diff,2023-03-13,-61.4800,-11.6783


### Interpreting the flagged observations

All ten flagged observations correspond to well-documented market events rather than data-entry errors, which is itself a useful confirmation of data quality.

**Brent, two days (2020-03-09, 2020-04-21):** the Saudi-Russia oil price war crash of March 2020, and the extreme volatility around the WTI negative-price episode in mid-to-late April 2020 that spilled over into Brent.

**DE2Y, two days (2023-03-13, 2023-03-15):** the collapse of Silicon Valley Bank triggered the largest one-day US 2-year Treasury yield drop since 2008 and the largest three-day drop since 1987, and the flight-to-quality rally pulled short-dated European yields down sharply in tandem; 15 March saw the same stress compound as the Credit Suisse crisis broke.

**JP2Y, three days (2016-07-29, 2020-03-27, 2024-08-05):** 29 July 2016 was a scheduled BoJ policy meeting at which the central bank only marginally expanded stimulus, disappointing markets that had priced in bolder easing; 5 August 2024 was the well-known global carry-trade unwind ("Black Monday"), triggered by a BoJ rate hike, during which JGB yields moved sharply alongside a broad cross-asset sell-off.

**Silver, one day (2020-08-11):** the sharp reversal following silver's rapid 2020 summer rally, widely referred to as the August 2020 silver flash crash.

**US2Y and US_JP_2Y_Diff, both on 2023-03-13:** the same SVB-driven collapse in US short-dated yields described above; the differential moves in lockstep because the shock originated almost entirely on the US side.

No observation is removed or corrected — all ten are retained as genuine extreme-volatility events, consistent with the crisis clustering already documented in the EDA notebook.

## 8. Export: wide panel and strict common-date panel

Two versions are saved. The **wide panel** keeps each series' own missing pattern intact — this is what pairwise analyses should read, so that one series' gaps (e.g. Brent's) never shrink the usable sample for an unrelated pair (e.g. EURUSD vs the US-DE differential). The **strict common-date panel** restricts to dates where every retained variable is observed simultaneously, for analyses that require one shared sample across the whole variable set (e.g. a joint multivariate model).

Missing values are never forward-filled at any stage: a forward-filled value would read as a zero change on that day and would distort the volatility proxies built on top of it.

In [85]:
wide = wide.sort_values("Date").reset_index(drop=True)

wide_path   = PROCESSED_DIR / "master_panel_wide.csv"
strict_path = PROCESSED_DIR / "master_panel_strict_common.csv"
summary_path = PROCESSED_DIR / "garch_egarch_summary.csv"

wide.to_csv(wide_path, index=False)
model_summary.to_csv(summary_path)

strict = wide.dropna(subset=[c for c in wide.columns if c != "Date"]).reset_index(drop=True)
strict.to_csv(strict_path, index=False)

print(f"Wide panel            : {wide.shape}   -> {wide_path.name}")
print(f"Strict common-date panel: {strict.shape} -> {strict_path.name}")
print(f"GARCH/EGARCH summary   : {model_summary.shape} -> {summary_path.name}")

Wide panel            : (2595, 116)   -> master_panel_wide.csv
Strict common-date panel: (1874, 116) -> master_panel_strict_common.csv
GARCH/EGARCH summary   : (15, 6) -> garch_egarch_summary.csv


In [86]:
required = [
    "EURUSD_dlogvol",
    "USDJPY_dlogvol",
    "AUDUSD_dlogvol",
    "USDCAD_dlogvol",
    "US_DE_2Y_Diff_dlogvol",
    "US_JP_2Y_Diff_dlogvol",
    "Brent_dlogvol",
    "Gold_dlogvol",
    "DXY_dlogvol",
]

print("Columns currently in memory:")
for col in required:
    print(f"{col:<30} {col in wide.columns}")

Columns currently in memory:
EURUSD_dlogvol                 True
USDJPY_dlogvol                 True
AUDUSD_dlogvol                 True
USDCAD_dlogvol                 True
US_DE_2Y_Diff_dlogvol          True
US_JP_2Y_Diff_dlogvol          True
Brent_dlogvol                  True
Gold_dlogvol                   True
DXY_dlogvol                    True


In [87]:
wide.to_csv(
    PROCESSED_DIR / "master_panel_wide.csv",
    index=True
)

print("Saved:", PROCESSED_DIR / "master_panel_wide.csv")
print("Shape:", wide.shape)

Saved: /Users/nancyshi/Desktop/2026 summer project/Volatility_Project/data/processed/master_panel_wide.csv
Shape: (2595, 116)


The strict common-date panel drops roughly 20% of the sample (2595 → 2088 rows) because it requires all 95 constructed columns — including robustness and derived proxy columns not needed in any single analysis — to be non-missing simultaneously, and the Japan-related cascading effect above disproportionately drives this loss. This exported version should be treated as a conservative reference case rather than the default input for any specific causality test: individual analyses should instead align only the handful of columns they actually use, so that one series' gaps never shrink the usable sample for an unrelated pair.

## 9. Summary

This notebook reads the four raw Bloomberg exports, restricts the sample to the series actually used in the design (four core FX pairs, six sovereign yields, Brent/Gold/Silver, JPM_FXVOL and VIX), cleans shared non-trading days and country-specific holiday staleness, and constructs returns, yield changes, and 2Y/10Y bilateral differentials for the two rates-driven pairs. It then builds four volatility representations — absolute, squared, GARCH(1,1) and EGARCH(1,1) conditional volatility — for every rates, commodity and FX innovation series, and exports both a wide panel (natural missingness, for pairwise analysis) and a strict common-date panel (for analyses needing one shared sample).

The next notebook (EDA) uses `master_panel_wide.csv` and `garch_egarch_summary.csv` to examine clustering, long memory, asymmetry and lead–lag structure in these volatility proxies.